# sebal-et-gee — Day 2 (v2): ERA5-Land + Net Radiation with Thrace-Wide Mosaic

**What changed from v1:** A single Landsat scene is ~170×185 km, but the Thrace AOI extends ~250 km east–west. We now mosaic multiple clear scenes within a 7-day window to cover the entire region, ensuring all 8 stations fall inside the footprint.

**Goal today (unchanged):** Compute net surface radiation (Rn) over the full Thrace AOI for a near-overpass date.

$$R_n = (1 - \alpha) \cdot R_{s\downarrow} + R_{l\downarrow} - R_{l\uparrow}$$

## 1. Setup

In [1]:
!pip install -q geemap geopandas

import ee
import geemap
import geopandas as gpd
import pandas as pd
from pathlib import Path

EE_PROJECT = 'et-thrace'  # <-- REPLACE with your project ID

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print('Earth Engine ready.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 16.9 MB/s eta 0:00:00
Earth Engine ready.


## 2. Load AOI from local GeoJSON + define 8 stations inline

Upload `thrace_boundary.geojson`. Stations are now hardcoded so the CSV is not required.

In [2]:
from google.colab import files

print('Upload thrace_boundary.geojson:')
uploaded = files.upload()

aoi_candidates = [f for f in uploaded if f.endswith(('.geojson', '.json'))]
if not aoi_candidates:
    raise FileNotFoundError('GeoJSON file not uploaded.')
aoi_path = aoi_candidates[0]
print(f'AOI file: {aoi_path}')


Upload thrace_boundary.geojson:


Saving thrace_stations.csv to thrace_stations.csv
Saving thrace_boundary.geojson to thrace_boundary.geojson
AOI file: thrace_boundary.geojson


In [3]:
aoi_gdf = gpd.read_file(aoi_path)
aoi_ee = geemap.geopandas_to_ee(aoi_gdf)
aoi_geom = aoi_ee.geometry()

# 8 stations from Scientific Reports manuscript (replace lat/lon with exact MGM values)
stations = pd.DataFrame([
    {'name': 'Edirne',      'lat': 41.6667, 'lon': 26.5667, 'elevation_m':  51},
    {'name': 'Kirklareli',  'lat': 41.7333, 'lon': 27.2167, 'elevation_m': 232},
    {'name': 'Luleburgaz',  'lat': 41.4000, 'lon': 27.3500, 'elevation_m':  46},
    {'name': 'Uzunkopru',   'lat': 41.2667, 'lon': 26.6833, 'elevation_m':  22},
    {'name': 'Ipsala',      'lat': 40.9167, 'lon': 26.3833, 'elevation_m':  10},
    {'name': 'Corlu',       'lat': 41.1500, 'lon': 27.8000, 'elevation_m': 183},
    {'name': 'Tekirdag',    'lat': 40.9833, 'lon': 27.5167, 'elevation_m':   3},
    {'name': 'Sariyer',     'lat': 41.1667, 'lon': 29.0500, 'elevation_m':  59},
])

station_features = [
    ee.Feature(ee.Geometry.Point([row['lon'], row['lat']]),
               {'name': row['name'], 'elevation_m': int(row['elevation_m'])})
    for _, row in stations.iterrows()
]
stations_ee = ee.FeatureCollection(station_features)

print(f'AOI area: {aoi_geom.area().divide(1e6).getInfo():,.0f} km²')
print(f'Stations loaded: {stations_ee.size().getInfo()}')


AOI area: 23,911 km²
Stations loaded: 8


## 3. Landsat path/row reconnaissance

Let's first find out which Landsat WRS-2 paths/rows cover Thrace. This helps us understand why we need a mosaic.

In [4]:
# Look at recent scenes intersecting the AOI and list their path/row
recon = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
         .filterBounds(aoi_geom)
         .filterDate('2023-06-01', '2023-09-30')
         .filter(ee.Filter.lt('CLOUD_COVER', 30)))

# Group by WRS path/row and show which ones intersect
def add_pathrow(img):
    path = img.get('WRS_PATH')
    row = img.get('WRS_ROW')
    return img.set('pathrow', ee.String(ee.Number(path).format()).cat('_').cat(ee.Number(row).format()))

recon = recon.map(add_pathrow)
distinct = recon.aggregate_array('pathrow').distinct().getInfo()
print(f'Distinct WRS-2 path/row tiles covering Thrace: {distinct}')
print(f'Total scenes in summer 2023: {recon.size().getInfo()}')


Distinct WRS-2 path/row tiles covering Thrace: ['180.0_31.0', '180.0_32.0', '181.0_31.0', '181.0_32.0', '181.0_33.0', '182.0_31.0', '182.0_32.0']
Total scenes in summer 2023: 39


## 4. Build the Thrace mosaic

Strategy: pick a **7-day window** in which clear scenes from all needed path/row tiles exist. Mosaic them. Keep each scene's own `system:time_start` for ERA5-Land matching.

We'll search for the best 7-day window by finding the anchor date with maximum AOI coverage and minimum cloud cover.

In [21]:
def apply_scale_factors(image):
    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermal_k = image.select('ST_B10').multiply(0.00341802).add(149.0).rename('LST_K')
    thermal_c = thermal_k.subtract(273.15).rename('LST_C')
    return image.addBands(optical, None, True).addBands(thermal_k).addBands(thermal_c)

def mask_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud = qa.bitwiseAnd(1 << 3).neq(0)
    shadow = qa.bitwiseAnd(1 << 4).neq(0)
    mask = cloud.Or(shadow).Not()
    return image.updateMask(mask)

COMMON_BANDS = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'ST_B10', 'QA_PIXEL']

def build_mosaic(anchor_str, window_days=14, max_cloud=80):
    anchor = ee.Date(anchor_str)
    start = anchor.advance(-window_days // 2, 'day')
    end = anchor.advance(window_days // 2, 'day')

    l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
          .filterBounds(aoi_geom).filterDate(start, end)
          .filter(ee.Filter.lt('CLOUD_COVER', max_cloud))
          .select(COMMON_BANDS))
    l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
          .filterBounds(aoi_geom).filterDate(start, end)
          .filter(ee.Filter.lt('CLOUD_COVER', max_cloud))
          .select(COMMON_BANDS))

    merged = (l8.merge(l9).map(mask_clouds).map(apply_scale_factors).sort('CLOUD_COVER'))
    mosaic = merged.mosaic().clip(aoi_geom)

    cov = mosaic.select('LST_K').mask().reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi_geom, scale=300, maxPixels=1e9
    ).getInfo().get('LST_K', 0) * 100

    return mosaic, merged, cov, start, end

# Try several summer 2023 anchors, pick the best
candidates = ['2023-06-25', '2023-07-10', '2023-07-20', '2023-08-05',
              '2023-08-10', '2023-08-20', '2023-09-01', '2023-09-10']

best = None
print('Searching best ANCHOR...')
for a in candidates:
    _, _, cov, _, _ = build_mosaic(a, window_days=14, max_cloud=80)
    print(f'  {a}: coverage = {cov:.1f}%')
    if best is None or cov > best[1]:
        best = (a, cov)

print(f'\n>>> Best ANCHOR: {best[0]} with coverage {best[1]:.1f}%')

ANCHOR = best[0]
WINDOW_DAYS = 14
mosaic, merged, coverage_pct, start, end = build_mosaic(ANCHOR, WINDOW_DAYS, max_cloud=80)

n_scenes = merged.size().getInfo()
print(f'\nTotal L8+L9 scenes in {WINDOW_DAYS}-day window around {ANCHOR}: {n_scenes}')
print(f'Final mosaic AOI coverage: {coverage_pct:.1f}%')

if coverage_pct < 80:
    print('⚠️ Coverage still insufficient. Trying window = 21...')
    mosaic, merged, coverage_pct, start, end = build_mosaic(ANCHOR, 21, max_cloud=80)
    WINDOW_DAYS = 21
    print(f'Window=21 coverage: {coverage_pct:.1f}%')

Searching best ANCHOR...
  2023-06-25: coverage = 99.7%
  2023-07-10: coverage = 99.4%
  2023-07-20: coverage = 99.7%
  2023-08-05: coverage = 99.3%
  2023-08-10: coverage = 52.5%
  2023-08-20: coverage = 97.4%
  2023-09-01: coverage = 95.1%
  2023-09-10: coverage = 73.7%

>>> Best ANCHOR: 2023-07-20 with coverage 99.7%

Total L8+L9 scenes in 14-day window around 2023-07-20: 14
Final mosaic AOI coverage: 99.7%


## 5. ERA5-Land: mosaic meteorology over the same window

Because different pixels come from different overpass times, we aggregate ERA5-Land within the ±WINDOW_DAYS/2 window at 10:00 UTC (approximate Landsat overpass for Thrace). This is a standard SEBAL simplification for composite/mosaic products.

In [22]:
era5_vars = [
    'temperature_2m',
    'dewpoint_temperature_2m',
    'u_component_of_wind_10m',
    'v_component_of_wind_10m',
    'surface_solar_radiation_downwards_hourly',
    'surface_thermal_radiation_downwards_hourly',
    'surface_pressure',
]

# Pick 10:00 UTC hour on each day in the window, then average
def era5_hour_at(date_obj, hour_utc=10):
    hour_start = date_obj.update(hour=hour_utc, minute=0, second=0)
    hour_end = hour_start.advance(1, 'hour')
    return (ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY')
            .filterDate(hour_start, hour_end)
            .select(era5_vars)
            .first())

# Build daily 10-UTC images for each date in the window
n_days = int(end.difference(start, 'day').getInfo())
daily_era5 = []
for i in range(n_days):
    d = start.advance(i, 'day')
    daily_era5.append(era5_hour_at(d, 10))

era5_coll = ee.ImageCollection(daily_era5)
era5_mean = era5_coll.mean().clip(aoi_geom)
print('ERA5-Land mean 10-UTC meteorology over the 7-day window computed.')
print('Bands:', era5_mean.bandNames().getInfo())


ERA5-Land mean 10-UTC meteorology over the 7-day window computed.
Bands: ['temperature_2m', 'dewpoint_temperature_2m', 'u_component_of_wind_10m', 'v_component_of_wind_10m', 'surface_solar_radiation_downwards_hourly', 'surface_thermal_radiation_downwards_hourly', 'surface_pressure']


## 6. Derive meteorology fields in SEBAL units

In [23]:
T_air_K = era5_mean.select('temperature_2m').rename('T_air_K')
T_air_C = T_air_K.subtract(273.15).rename('T_air_C')

u10 = era5_mean.select('u_component_of_wind_10m')
v10 = era5_mean.select('v_component_of_wind_10m')
wind_10m = u10.pow(2).add(v10.pow(2)).sqrt().rename('wind_10m')

Rs_in = era5_mean.select('surface_solar_radiation_downwards_hourly').divide(3600).rename('Rs_in')
Rl_in = era5_mean.select('surface_thermal_radiation_downwards_hourly').divide(3600).rename('Rl_in')

P_surf = era5_mean.select('surface_pressure').rename('P_surf')

meteo = ee.Image.cat([T_air_K, T_air_C, wind_10m, Rs_in, Rl_in, P_surf])

# Sanity check at stations
station_meteo = meteo.reduceRegions(
    collection=stations_ee,
    reducer=ee.Reducer.first(),
    scale=9000,
)
df = geemap.ee_to_df(station_meteo)[['name', 'T_air_C', 'wind_10m', 'Rs_in', 'Rl_in', 'P_surf']].round(2)
print('Station meteorology (10 UTC mean over window):')
df


Station meteorology (10 UTC mean over window):


,name,T_air_C,wind_10m,Rs_in,Rl_in,P_surf
0,Edirne,34.75,0.92,879.48,412.37,100245.23
1,Kirklareli,31.07,1.13,872.70,397.50,98319.16
2,Luleburgaz,34.02,1.90,887.45,404.39,100123.95
3,Uzunkopru,35.13,1.32,889.78,411.80,100603.09
4,Ipsala,34.43,1.37,893.61,405.46,100619.80
5,Corlu,32.28,1.93,892.55,397.69,100102.52
6,Tekirdag,NaN,NaN,NaN,NaN,NaN
7,Sariyer,31.59,2.73,889.96,393.33,100311.09


## 7. Outgoing longwave from mosaic LST

In [24]:
SIGMA = 5.670374419e-8

ndvi = mosaic.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
emissivity = ndvi.multiply(0.01).add(0.985).clamp(0.96, 0.99).rename('emissivity')

LST_K = mosaic.select('LST_K')
Rl_out = emissivity.multiply(SIGMA).multiply(LST_K.pow(4)).rename('Rl_out')
print('Rl_out computed from mosaic LST.')


Rl_out computed from mosaic LST.


## 8. Net Radiation with placeholder albedo

In [25]:
ALBEDO_PLACEHOLDER = 0.23

Rn = (Rs_in.multiply(1 - ALBEDO_PLACEHOLDER)
      .add(Rl_in)
      .subtract(Rl_out)
      .rename('Rn'))
print(f'Rn computed with placeholder albedo = {ALBEDO_PLACEHOLDER}.')


Rn computed with placeholder albedo = 0.23.


## 9. Visualize the full-Thrace mosaic + Rn

In [26]:
m = geemap.Map(center=[41.3, 27.5], zoom=8)
m.add_basemap('SATELLITE')

m.addLayer(mosaic, {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 0.3},
           f'Landsat 9 Mosaic ({ANCHOR} ±{WINDOW_DAYS // 2}d)')

m.addLayer(ndvi, {'min': -0.1, 'max': 0.9, 'palette': ['blue', 'white', 'green']},
           'NDVI', False)

m.addLayer(mosaic.select('LST_C'),
           {'min': 15, 'max': 45,
            'palette': ['040274', '0502ff', '5dfff0', 'ffc900', 'ff0000', '800080']},
           'LST °C', False)

m.addLayer(Rn, {'min': 300, 'max': 800,
                'palette': ['000080', '0000ff', '00ffff', 'ffff00', 'ff8000', 'ff0000']},
           'Net Radiation Rn (W/m²)')

m.addLayer(aoi_ee, {'color': 'yellow'}, 'Thrace AOI')
m.addLayer(stations_ee, {'color': 'red'}, '8 stations')

m.addLayerControl()
m


Map(center=[41.3, 27.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

## 10. Rn sanity check at all 8 stations

All stations should now return valid (non-NaN) values in the 450–750 W/m² range.

In [27]:
station_rn = Rn.reduceRegions(
    collection=stations_ee,
    reducer=ee.Reducer.mean(),
    scale=30,
)
df_rn = geemap.ee_to_df(station_rn)[['name', 'mean']].rename(columns={'mean': 'Rn_Wm2'})
df_rn['Rn_Wm2'] = df_rn['Rn_Wm2'].round(1)
print('Station-level Rn (W/m²):')
df_rn


Station-level Rn (W/m²):


,name,Rn_Wm2
0,Edirne,488.3
1,Kirklareli,523.0
2,Luleburgaz,513.2
3,Uzunkopru,513.7
4,Ipsala,556.9
5,Corlu,504.0
6,Tekirdag,512.8
7,Sariyer,559.6


## 11. Day 2 v2 deliverables checklist

- [ ] Mosaic AOI coverage > 95%
- [ ] All 8 stations return valid Rn values
- [ ] Rn values in expected 450–750 W/m² range
- [ ] Heterogeneous Rn pattern visible on the map (ag land high, water/urban low)

If coverage is still partial, try a different `ANCHOR` date (e.g., `'2023-07-20'`, `'2023-08-25'`) or increase `WINDOW_DAYS` to 14.

**Day 3 preview:** replace the placeholder albedo with a true Landsat-derived broadband albedo (Tasumi 2008 coefficients) and compute soil heat flux G.